In [ ]:
# 背景：在09文件跑逻辑回归模型后，发现在保证字段干净的情况下模型性能不太好。
# 但是我们学习逻辑回归模型的目的已经达到，现在开始尝试品牌四象限矩阵。

# 本次目标：构建品牌四象限矩阵，分析品牌在不同象限的分布情况。
# 注意：本分析只使用 A/C 明确结果样本，不包含 B 组。
# 因此这里的“规模”“购买占比”“流失占比”都只代表 A/C 明确结果样本内部的相对表现，
# 不能解释成品牌在全量加购用户中的真实整体转化率。

# 四象限使用两个核心字段：
# - x轴：brand_share_in_clear_outcome（品牌在 A/C 明确结果样本中的体量占比）
# - y轴：c_to_a_ratio（C_count / A_count，主动流失相对购买的风险倍数）
#
# 四个象限：
# - 高体量 × 高风险：重点治理品牌
# - 高体量 × 低风险：核心健康品牌
# - 低体量 × 高风险：问题长尾品牌
# - 低体量 × 低风险：潜力/普通长尾品牌

# 字段工程伪代码模板：
# 1. 读取本地数据库 pgsql 中的 makeup_consumer_events."03_user_behavior_groups" 表，得到 df
# 2. 筛选 group_type 字段为 "A" 和 "C" 的行，得到 df_ac
# 3. 在 groupby 前处理 brand 缺失：
#    - df_ac["brand_clean"] = df_ac["brand"].fillna("Unknown Brand")
#    - 如果存在空字符串，也统一替换为 "Unknown Brand"
# 4. 按 brand_clean 聚合，得到以下字段：
#    - A_count = 该品牌 A 组记录数
#    - C_count = 该品牌 C 组记录数
#    - clear_outcome_count = A_count + C_count
#    - ac_purchase_share = A_count / clear_outcome_count
#    - ac_loss_share = C_count / clear_outcome_count
#    - c_to_a_ratio = C_count / A_count
#    - brand_share_in_clear_outcome = clear_outcome_count / clear_outcome_count.sum()
#
# 字段含义提醒：
# - clear_outcome_count 不是品牌总样本量，而是该品牌在 A/C 明确结果样本中的记录数
# - ac_purchase_share 不是全量购买率，而是 A/C 明确结果中 A 组占比
# - ac_loss_share 不是全量流失率，而是 A/C 明确结果中 C 组占比
# - c_to_a_ratio 越高，说明该品牌相对购买更偏向主动流失

# 特殊情况：
# 1. 仅使用 A/C 明确结果样本
# 2. brand 缺失统一标记为 Unknown Brand
# 3. Unknown Brand 不进入四象限主图，单独统计
# 4. clear_outcome_count < 100 的品牌不进入四象限主图，归为样本不足品牌
# 5. 如果 A_count = 0，c_to_a_ratio 会无法计算或变成无穷大：
#    - 这类品牌不直接进入主图
#    - 单独列为“只有主动流失、没有购买记录”的高风险观察品牌
# 6. 四象限主图只分析：
#    brand != Unknown Brand
#    clear_outcome_count >= 100
#    A_count > 0

# 四象限切分方法（第一版先用中位数，简单且便于解释）：
# - share_threshold = qualified_brands["brand_share_in_clear_outcome"].median()
# - risk_threshold = qualified_brands["c_to_a_ratio"].median()
# - brand_share_in_clear_outcome >= share_threshold 记为高体量，否则低体量
# - c_to_a_ratio >= risk_threshold 记为高风险，否则低风险
#
# 后续如果想更业务化，可以把切分方法改成：
# - 规模：按品牌累计贡献 Top 80% 区分核心品牌/长尾品牌
# - 风险：用 c_to_a_ratio > 1 判断主动流失是否高于购买
